# Fast sentiment on CPU — MiniLM-L6 + int8
Local, no GPU needed for inference. Put `train.csv` and `test.csv` (columns: `text`, `label` 0/1) next to this notebook.

In [1]:
!pip install -q -U transformers datasets accelerate scikit-learn

In [2]:
import os, torch, numpy as np, pandas as pd
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, f1_score

torch.set_num_threads(os.cpu_count())
MODEL = "nreimers/MiniLM-L6-H384-uncased"   # 22M params, 6 layers
MAX_LEN = 256                                # lower this if your texts are short
print("device:", "cuda" if torch.cuda.is_available() else "cpu", "| threads:", os.cpu_count())

device: cuda | threads: 20


In [4]:
train = pd.read_csv("../raw_data/train.csv")
test  = pd.read_csv("../raw_data/test.csv")
print("train:", train.shape, "| test:", test.shape)

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)

def to_ds(df):
    ds = Dataset.from_pandas(df[["text", "label"]], preserve_index=False)
    return ds.rename_column("label", "labels")

def tokenize(b): return tokenizer(b["text"], truncation=True, max_length=MAX_LEN)

train_ds = to_ds(train).map(tokenize, batched=True)
test_ds  = to_ds(test).map(tokenize, batched=True)

train: (15664, 2) | test: (3917, 2)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15213.88it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: nreimers/MiniLM-L6-H384-uncased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Map: 100%|██████████| 3917/3917 [00:00<00:00, 23197.00 examples/s]


In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}

args = TrainingArguments(
    output_dir="minilm_ft",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_ds, eval_dataset=test_ds,
                  compute_metrics=compute_metrics,
                  data_collator=DataCollatorWithPadding(tokenizer))

Training is much faster on a GPU (e.g. Colab T4). If you train on CPU it works but is slow — you only need CPU speed for **inference**, which is tested below.

In [8]:
trainer.train()
print(trainer.evaluate())

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.138360,0.192337,0.949196,0.933564
2,0.099850,0.133108,0.964003,0.954582
3,0.059955,0.148321,0.966045,0.957358


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 16.28it/s]


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1
0.059955,0.148321,3,0.966045,0.957358


{'eval_loss': 0.1483210325241089, 'eval_accuracy': 0.9660454429410263, 'eval_macro_f1': 0.9573579251645137}


In [9]:
trainer.save_model("minilm_ft")
tokenizer.save_pretrained("minilm_ft")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 14.57it/s]


('minilm_ft/tokenizer_config.json', 'minilm_ft/tokenizer.json')

## CPU inference benchmark
Loads on CPU then times batched prediction and extrapolates to 11k articles.

In [5]:
import time

cpu_model = AutoModelForSequenceClassification.from_pretrained("minilm_ft").to("cpu").eval()

texts = test["text"].tolist()

def predict(m, texts, batch_size=64):
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i+batch_size], truncation=True,
                        max_length=MAX_LEN, padding=True, return_tensors="pt")
        with torch.no_grad():
            logits = m(**enc).logits
        out.extend(logits.argmax(-1).tolist())
    return out

def bench(m, name):
    t = time.time(); predict(m, texts); dt = time.time() - t
    rate = len(texts) / dt
    print(f"{name:>10}: {dt:5.1f}s for {len(texts)} texts | {rate:6.1f}/s | 11k ≈ {11000/rate/60:4.1f} min")

bench(cpu_model, "fp32")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3449.56it/s]


      fp32:  45.8s for 3917 texts |   85.4/s | 11k ≈  2.1 min
